**Data note.** The original data files (Kaggle `train.csv` and `test.csv`, which need a Kaggle login) are not in this repository; this notebook now downloads the Titanic passenger list from OpenML (data set 40945, authors Frank E. Harrell Jr. and Thomas Cason, Vanderbilt Biostatistics; 1,309 passengers), OpenML licence field "Public", instead; results differ from the earlier outputs. The rows are shuffled with a fixed seed and split 891 / 418, and the columns are given the Kaggle names. Because the 418 held-out rows keep their labels here, one cell was added at the end of the model comparison to score the models on them. The Kaggle boilerplate that listed `/kaggle/input` was removed.

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Data: the Titanic passenger list from OpenML (data set 40945: Harrell and Cason, Vanderbilt Biostatistics).
# It is shuffled with a fixed seed and split into 891 training and 418 test rows (the sizes of the Kaggle files),
# and given the Kaggle column names so the rest of the notebook runs unchanged.
# The columns "boat", "body" and "home.dest" are left out (boat and body reveal who survived).
import os, time
from sklearn.datasets import fetch_openml

os.makedirs("data", exist_ok=True)
if not os.path.exists("data/titanic_train.csv"):
    for attempt in range(4):  # the OpenML server sometimes refuses a request; retry a few times
        try:
            raw = fetch_openml(data_id=40945, as_frame=True, data_home="data/openml").frame
            break
        except Exception as err:
            print("OpenML request failed, retrying:", err)
            time.sleep(10)
    else:
        raise RuntimeError("OpenML download failed")
    raw = raw.rename(columns={"pclass": "Pclass", "survived": "Survived", "name": "Name", "sex": "Sex", "age": "Age",
                              "sibsp": "SibSp", "parch": "Parch", "ticket": "Ticket", "fare": "Fare",
                              "cabin": "Cabin", "embarked": "Embarked"})
    raw = raw.sample(frac=1, random_state=42).reset_index(drop=True)
    raw.insert(0, "PassengerId", range(1, len(raw) + 1))
    raw = raw[["PassengerId", "Survived", "Pclass", "Name", "Sex", "Age", "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked"]]
    for col in ["Survived", "Pclass"]:
        raw[col] = raw[col].astype(int)
    for col in ["Name", "Sex", "Ticket", "Cabin", "Embarked"]:
        raw[col] = raw[col].astype(object)
    raw.iloc[:891].to_csv("data/titanic_train.csv", index=False)
    raw.iloc[891:].drop(columns="Survived").to_csv("data/titanic_test.csv", index=False)
    raw.iloc[891:][["PassengerId", "Survived"]].to_csv("data/titanic_test_labels.csv", index=False)
print(pd.read_csv("data/titanic_train.csv").shape, pd.read_csv("data/titanic_test.csv").shape)

...\site-packages\sklearn\datasets\_openml.py:107: UserWarning: A network error occurred while downloading https://api.openml.org/api/v1/json/data/40945. Retrying...
  warn(


OpenML request failed, retrying: HTTP Error 503: Service Temporarily Unavailable


(891, 12) (418, 11)


In [3]:
train_data = pd.read_csv("data/titanic_train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Rintamaki, Mr. Matti",male,35.0,0,0,STON/O 2. 3101273,7.1250,NaN,S
1,2,1,3,"Nakid, Mr. Sahid",male,20.0,1,1,2653,15.7417,NaN,C
2,3,0,3,"Lyntakoff, Mr. Stanko",male,NaN,0,0,349235,7.8958,NaN,S
3,4,0,3,"Ford, Mr. Arthur",male,NaN,0,0,A/5 1478,8.0500,NaN,S
4,5,0,3,"Shaughnessy, Mr. Patrick",male,NaN,0,0,370374,7.7500,NaN,Q


In [4]:
test_data = pd.read_csv("data/titanic_test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,1,"Nicholson, Mr. Arthur Ernest",male,64.0,0,0,693,26.0000,NaN,S
1,893,1,"Meyer, Mr. Edgar Joseph",male,28.0,1,0,PC 17604,82.1708,NaN,C
2,894,3,"Nenkoff, Mr. Christo",male,NaN,0,0,349234,7.8958,NaN,S
3,895,3,"Novel, Mr. Mansouer",male,28.5,0,0,2697,7.2292,NaN,C
4,896,1,"Dodge, Dr. Washington",male,53.0,1,1,33638,81.8583,A34,S


In [5]:
train_data.describe()


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,713.000000,891.000000,891.000000,890.000000
mean,446.000000,0.383838,2.277217,29.851683,0.460157,0.350168,33.486783
std,257.353842,0.486592,0.833651,14.479544,1.020760,0.747090,52.941276
min,1.000000,0.000000,1.000000,0.416700,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,21.000000,0.000000,0.000000,7.895800
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,13.879150
75%,668.500000,1.000000,3.000000,39.000000,1.000000,0.000000,30.500000
max,891.000000,1.000000,3.000000,76.000000,8.000000,6.000000,512.329200


In [6]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Rintamaki, Mr. Matti",male,35.0,0,0,STON/O 2. 3101273,7.1250,NaN,S
1,2,1,3,"Nakid, Mr. Sahid",male,20.0,1,1,2653,15.7417,NaN,C
2,3,0,3,"Lyntakoff, Mr. Stanko",male,NaN,0,0,349235,7.8958,NaN,S
3,4,0,3,"Ford, Mr. Arthur",male,NaN,0,0,A/5 1478,8.0500,NaN,S
4,5,0,3,"Shaughnessy, Mr. Patrick",male,NaN,0,0,370374,7.7500,NaN,Q


In [7]:
train_data = train_data.drop(['Ticket', 'Cabin'], axis=1)
test_data= test_data.drop(['Ticket', 'Cabin'], axis=1)
combine = [train_data, test_data]
for dataset in combine:
    dataset['Title'] = dataset.Name.str.extract(' ([A-Za-z]+)\.', expand=False)

In [8]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,1,0,3,"Rintamaki, Mr. Matti",male,35.0,0,0,7.1250,S,Mr
1,2,1,3,"Nakid, Mr. Sahid",male,20.0,1,1,15.7417,C,Mr
2,3,0,3,"Lyntakoff, Mr. Stanko",male,NaN,0,0,7.8958,S,Mr
3,4,0,3,"Ford, Mr. Arthur",male,NaN,0,0,8.0500,S,Mr
4,5,0,3,"Shaughnessy, Mr. Patrick",male,NaN,0,0,7.7500,Q,Mr


In [9]:
train_data = train_data.drop(['Name', 'PassengerId'], axis=1)
test_data = test_data.drop(['Name'], axis=1)

train_data.shape, test_data.shape

((891, 9), (418, 9))

In [10]:
combine = [train_data, test_data]
for dataset in combine:
    dataset['Sex'] = dataset['Sex'].map( {'female': 1, 'male': 0} ).astype(int)

train_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,0,3,0,35.0,0,0,7.1250,S,Mr
1,1,3,0,20.0,1,1,15.7417,C,Mr
2,0,3,0,NaN,0,0,7.8958,S,Mr
3,0,3,0,NaN,0,0,8.0500,S,Mr
4,0,3,0,NaN,0,0,7.7500,Q,Mr


In [11]:
for dataset in combine:
    dataset['Title'] = dataset['Title'].replace(['Lady', 'Countess','Capt', 'Col',\
 	'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')

    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Ms', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme', 'Mrs')
    
train_data[['Title', 'Survived']].groupby(['Title'], as_index=False).mean()
    
   
    

,Title,Survived
0,Master,0.575000
1,Miss,0.697802
2,Mr,0.153551
3,Mrs,0.818898
4,Rare,0.380952


In [12]:
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
for dataset in combine:
    dataset['Title'] = dataset['Title'].map(title_mapping)
    dataset['Title'] = dataset['Title'].fillna(0)

train_data.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,0,3,0,35.0,0,0,7.1250,S,1
1,1,3,0,20.0,1,1,15.7417,C,1
2,0,3,0,NaN,0,0,7.8958,S,1
3,0,3,0,NaN,0,0,8.0500,S,1
4,0,3,0,NaN,0,0,7.7500,Q,1


In [13]:
train_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,0,3,0,35.0,0,0,7.1250,S,1
1,1,3,0,20.0,1,1,15.7417,C,1
2,0,3,0,NaN,0,0,7.8958,S,1
3,0,3,0,NaN,0,0,8.0500,S,1
4,0,3,0,NaN,0,0,7.7500,Q,1


In [14]:
freq_port = train_data.Embarked.dropna().mode()[0]
freq_port

'S'

In [15]:
for dataset in combine:
    dataset['Embarked'] = dataset['Embarked'].fillna(freq_port)

In [16]:
combine = [train_data, test_data]
for dataset in combine:
    dataset['Embarked'] = dataset['Embarked'].map( {'S': 0, 'C': 1, 'Q': 2} ).astype(int)

train_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,0,3,0,35.0,0,0,7.1250,0,1
1,1,3,0,20.0,1,1,15.7417,1,1
2,0,3,0,NaN,0,0,7.8958,0,1
3,0,3,0,NaN,0,0,8.0500,0,1
4,0,3,0,NaN,0,0,7.7500,2,1


In [17]:
f=train_data.Age.dropna().mean()

f
g=train_data.Fare.dropna().mean()
g

33.48678303370786

In [18]:

  
# displaying the datatypes
display(dataset['Age'].dtypes)

dtype('float64')

In [19]:
for dataset in combine:
    dataset['Fare'] = dataset['Fare'].fillna(g)
   
    dataset['Fare']= dataset['Fare'].astype(int)




In [20]:
for dataset in combine:
    dataset['Age'] = dataset['Age'].fillna(f)
    dataset['Age']= dataset['Age'].astype(int)

In [21]:
test_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   PassengerId  418 non-null    int64
 1   Pclass       418 non-null    int64
 2   Sex          418 non-null    int32
 3   Age          418 non-null    int32
 4   SibSp        418 non-null    int64
 5   Parch        418 non-null    int64
 6   Fare         418 non-null    int32
 7   Embarked     418 non-null    int32
 8   Title        418 non-null    int64
dtypes: int32(4), int64(5)
memory usage: 23.0 KB


In [22]:
test_data['Fare']= test_data['Fare'].astype(int)

In [23]:

train_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,0,3,0,35,0,0,7,0,1
1,1,3,0,20,1,1,15,1,1
2,0,3,0,29,0,0,7,0,1
3,0,3,0,29,0,0,8,0,1
4,0,3,0,29,0,0,7,2,1


In [24]:
women = train_data.loc[train_data.Sex == 1]["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

% of women who survived: 0.75


In [25]:
men = train_data.loc[train_data.Sex == 0]["Survived"]
print("% of men",men)
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of men 0      0
1      1
2      0
3      0
4      0
      ..
885    0
886    1
887    0
889    0
890    0
Name: Survived, Length: 579, dtype: int64
% of men who survived: 0.18652849740932642


In [26]:
X_train = train_data.drop("Survived", axis=1)
Y_train = train_data["Survived"]
X_test  = test_data.drop("PassengerId", axis=1).copy()
X_train.shape, Y_train.shape, X_test.shape

((891, 8), (891,), (418, 8))

In [27]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

In [28]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(n_estimators=100)
random_forest.fit(X_train, Y_train)
Y_pred = random_forest.predict(X_test)
random_forest.score(X_train, Y_train)
acc_random_forest = round(random_forest.score(X_train, Y_train) * 100, 2)
acc_random_forest

95.96

In [29]:
# Logistic Regression

logreg = LogisticRegression()
logreg.fit(X_train, Y_train)
Y_pred = logreg.predict(X_test)
acc_log = round(logreg.score(X_train, Y_train) * 100, 2)
acc_log

...\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


81.93

In [30]:
coeff_df = pd.DataFrame(train_data.columns.delete(0))
coeff_df.columns = ['Feature']
coeff_df["Correlation"] = pd.Series(logreg.coef_[0])

coeff_df.sort_values(by='Correlation', ascending=False)

,Feature,Correlation
1,Sex,2.147222
7,Title,0.511320
6,Embarked,0.369759
5,Fare,0.003196
2,Age,-0.032957
4,Parch,-0.059963
3,SibSp,-0.425769
0,Pclass,-0.900494


In [31]:
from sklearn.svm import SVC, LinearSVC
svc = SVC()
svc.fit(X_train, Y_train)
Y_pred = svc.predict(X_test)
acc_svc = round(svc.score(X_train, Y_train) * 100, 2)
acc_svc

67.68

In [32]:
# Linear SVC

linear_svc = LinearSVC()
linear_svc.fit(X_train, Y_train)
Y_pred = linear_svc.predict(X_test)
acc_linear_svc = round(linear_svc.score(X_train, Y_train) * 100, 2)
acc_linear_svc

81.82

In [33]:
decision_tree = DecisionTreeClassifier()
decision_tree.fit(X_train, Y_train)
Y_pred = decision_tree.predict(X_test)
acc_decision_tree = round(decision_tree.score(X_train, Y_train) * 100, 2)
acc_decision_tree

95.96

In [34]:
sgd = SGDClassifier()
sgd.fit(X_train, Y_train)
Y_pred = sgd.predict(X_test)
acc_sgd = round(sgd.score(X_train, Y_train) * 100, 2)
acc_sgd

68.91

In [35]:

gaussian = GaussianNB()
gaussian.fit(X_train, Y_train)
Y_pred = gaussian.predict(X_test)
acc_gaussian = round(gaussian.score(X_train, Y_train) * 100, 2)
acc_gaussian

80.36

In [36]:
knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(X_train, Y_train)
Y_pred = knn.predict(X_test)
acc_knn = round(knn.score(X_train, Y_train) * 100, 2)
acc_knn

83.61

In [37]:
perceptron = Perceptron()
perceptron.fit(X_train, Y_train)
Y_pred = perceptron.predict(X_test)
acc_perceptron = round(perceptron.score(X_train, Y_train) * 100, 2)
acc_perceptron

65.32

In [38]:
models = pd.DataFrame({
    'Model': ['Support Vector Machines', 'KNN', 'Logistic Regression', 
              'Random Forest', 'Naive Bayes', 'Perceptron', 
              'Stochastic Gradient Decent', 'Linear SVC', 
              'Decision Tree'],
    'Score': [acc_svc, acc_knn, acc_log, 
              acc_random_forest, acc_gaussian, acc_perceptron, 
              acc_sgd, acc_linear_svc, acc_decision_tree]})
models.sort_values(by='Score', ascending=False)

,Model,Score
3,Random Forest,95.96
8,Decision Tree,95.96
1,KNN,83.61
2,Logistic Regression,81.93
7,Linear SVC,81.82
4,Naive Bayes,80.36
6,Stochastic Gradient Decent,68.91
0,Support Vector Machines,67.68
5,Perceptron,65.32


In [39]:
# Added for the substitute data: the 418 held-out rows come with labels, so each model can also be scored on them.
# (The scores above are training-set accuracy, as in the original notebook.)
test_labels = pd.read_csv("data/titanic_test_labels.csv")["Survived"]
fitted = {"Support Vector Machines": svc, "KNN": knn, "Logistic Regression": logreg, "Random Forest": random_forest,
          "Naive Bayes": gaussian, "Perceptron": perceptron, "Stochastic Gradient Decent": sgd,
          "Linear SVC": linear_svc, "Decision Tree": decision_tree}
pd.DataFrame({"Model": list(fitted), "Held-out accuracy": [round(m.score(X_test, test_labels) * 100, 2) for m in fitted.values()]}
             ).sort_values(by="Held-out accuracy", ascending=False)

,Model,Held-out accuracy
2,Logistic Regression,77.99
3,Random Forest,77.27
7,Linear SVC,76.56
4,Naive Bayes,75.84
8,Decision Tree,74.16
1,KNN,70.57
0,Support Vector Machines,67.70
6,Stochastic Gradient Decent,66.27
5,Perceptron,63.64


In [40]:
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': Y_pred})

output.to_csv('my_submission.csv', index=False)

In [41]:
output.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0


In [42]:
#submission.to_csv('submission.csv', index=False)